# 03 — Model Training

Trains a binary image classifier using EfficientNetB0 pretrained on ImageNet.

**Two-phase training:**
1. Phase 1 — freeze the EfficientNetB0 base, train only the classification head (10 epochs)
2. Phase 2 — unfreeze the last 30 layers, fine-tune at a lower learning rate (10 epochs)

Saves the best model as `.keras` and converts to ONNX for serving.

**Expected time on Kaggle T4 GPU: ~3–4 hours for the full PlantNet dataset.**

In [ ]:
import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

print('TensorFlow version:', tf.__version__)
print('GPUs available:', tf.config.list_physical_devices('GPU'))

IMG_SIZE   = 224
BATCH_SIZE = 32

In [ ]:
# --- Load dataset splits from Notebook 01 ---
train_df = pd.read_csv('/kaggle/working/train.csv')
val_df   = pd.read_csv('/kaggle/working/val.csv')
print(f'Train: {len(train_df)}  |  Val: {len(val_df)}')

In [ ]:
# --- Build tf.data pipeline ---
def make_dataset(df, augment=False):
    paths  = df['path'].values
    labels = df['label'].values.astype('float32')

    def load_and_preprocess(path, label):
        img = tf.io.read_file(path)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
        img = preprocess_input(img)  # scales to [-1, 1] — must match app.py
        return img, label

    def augment_fn(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.random_saturation(img, 0.8, 1.2)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.shuffle(buffer_size=10000, reshuffle_each_iteration=True)  # prevents order bias
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, augment=True)
val_ds   = make_dataset(val_df,   augment=False)

In [ ]:
# --- Build model ---
# Input named 'input' explicitly for ONNX tensor name compatibility
inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='input')
base   = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(256, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model = tf.keras.Model(inputs, outputs)

model.summary()

In [ ]:
# --- Phase 1: Train classification head only (base frozen) ---
model.compile(
    optimizer=Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

history1 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=10,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
)
print('Phase 1 complete')

In [ ]:
# --- Phase 2: Unfreeze last 30 layers for fine-tuning ---
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

# Lower learning rate to avoid destroying pretrained features
model.compile(
    optimizer=Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

history2 = model.fit(
    train_ds, validation_data=val_ds,
    epochs=10,
    callbacks=[
        EarlyStopping(patience=3, restore_best_weights=True),
        ModelCheckpoint('/kaggle/working/nature_classifier.keras', save_best_only=True)
    ]
)
print('Phase 2 complete')

In [ ]:
# --- Plot training curves ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Combine both phases
loss     = history1.history['loss']     + history2.history['loss']
val_loss = history1.history['val_loss'] + history2.history['val_loss']
auc      = history1.history['auc']      + history2.history['auc']
val_auc  = history1.history['val_auc']  + history2.history['val_auc']

axes[0].plot(loss, label='Train loss')
axes[0].plot(val_loss, label='Val loss')
axes[0].axvline(len(history1.history['loss']), color='gray', linestyle='--', label='Fine-tune start')
axes[0].set_title('Loss')
axes[0].legend()

axes[1].plot(auc, label='Train AUC')
axes[1].plot(val_auc, label='Val AUC')
axes[1].axvline(len(history1.history['auc']), color='gray', linestyle='--', label='Fine-tune start')
axes[1].set_title('AUC')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# --- Convert to ONNX ---
import tf2onnx

model_proto, _ = tf2onnx.convert.from_keras(
    model,
    input_signature=[tf.TensorSpec((None, IMG_SIZE, IMG_SIZE, 3), tf.float32, name='input')]
)
onnx_path = '/kaggle/working/nature_classifier.onnx'
with open(onnx_path, 'wb') as f:
    f.write(model_proto.SerializeToString())
print(f'ONNX model saved to {onnx_path}')

# Verify ONNX input tensor name — must match what app.py uses
import onnxruntime as ort
session = ort.InferenceSession(onnx_path)
print(f'ONNX input tensor name: "{session.get_inputs()[0].name}"')  # should print "input"